In [1]:
import trodes.read_exported as tr
import os
import json
import numpy as np
import pickle

In [2]:
# insert path to folder with videoTimestamps files and merged.time folders for each recording
path = r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps"


In [4]:
# use this cell to extract first timestamp from each recording 
# find a similar pattern between merged.time folder names and videotimestamps file names
# in this case the date is the smae across both file types

# side note: first timestamp per recording in a recording session are the same 
# ei if you recorded from 4 headstages at the same time (all in the same rec folder)
# each one will have the same first timestamp 

first_timestamp_dict = {}
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".dat"):
            ts_file = os.path.join(root, file)
            ts_dict = tr.read_trodes_extracted_data_file(ts_file)
            first_timestamp = ts_dict['first_timestamp']
            file_parts = file.split('_')
            file_pattern = file_parts[0] + '_' + file_parts[1]
            first_timestamp_dict[file_pattern] = int(first_timestamp)

print(first_timestamp_dict)

with open('pilot2/object_control/first_timestamps_object_control','wb') as file:
        pickle.dump(first_timestamp_dict, file)

{'20250618_113931': 1959619, '20250618_123431': 1665815, '20250618_133349': 2437744, '20250618_143034': 1243125, '22_object': 1768370, '23_object': 2159185, '31_object': 2038615, '32_object': 2038615, '41_object': 2159185, '44_object': 1768370}


In [9]:
# this cell will create a dictionary with three items under each merged.rec file name
# stream_indexed_array: camera module timestamps in seconds where the 0th second is at stream
# play_indexed_array: camera module timestamps in seconds where the 0th second is at play (when you clicked play in trodes)
# first_timestamp: the timestamp in 20KHz of when you clicked play, where the 0th timestamp is at stream

play_indexed_dict = {}
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".videoTimeStamps"):
            play_indexed_dict[file] = {}
            print(os.path.join(root, file))
            videotsfile = os.path.join(root, file)
            videotsarray = tr.readCameraModuleTimeStamps(videotsfile)
            date = file.split('_')[0] + '_' + file.split('_')[1]
            try:
                first_ts = first_timestamp_dict[date]
                print(date)
            except KeyError:
                for key in first_timestamp_dict.keys():
                    if date.split('_')[0] in key:
                        print(key)
                        first_ts = first_timestamp_dict[key]
            videotsarray_play_indexed = videotsarray - first_ts/20000
            if videotsarray_play_indexed[0] < 0:
                if abs(videotsarray_play_indexed[0]) < .001:
                    videotsarray_play_indexed[0] = 0
                else:
                    print('negative first timestamp: ', videotsarray_play_indexed[0])
            play_indexed_dict[file]['play_indexed_array'] = videotsarray_play_indexed
            play_indexed_dict[file]['first_timestamp'] = first_ts
            play_indexed_dict[file]['stream_indexed_array'] =  videotsarray 




C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps\20250618_113931_object_control_subj_1-2_and_6-1.1.videoTimeStamps
20250618_113931
C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps\20250618_113931_object_control_subj_1-2_and_6-1.2.videoTimeStamps
20250618_113931
C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps\20250618_123431_object_control_subj_1-3_and_6-3.1.videoTimeStamps
20250618_123431
C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps\20250618_123431_object_control_subj_1-3_and_6-3.2.videoTimeStamps
20250618_123431
C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Object_Control (phase 7)\videoTimeStamps\20250618_133349_object_cont

In [10]:
# this function saves the dictionary to a json to be used later when creating an event dict
# that is accurately aligned to the neural data timestamps

def convert_numpy_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_to_list(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_list(item) for item in obj]
    else:
        return obj

# Convert the dictionary for JSON serialization
json_compatible_dict = convert_numpy_to_list(play_indexed_dict)

# Save to JSON file
output_filename = 'pilot2/object_control/timestamps_dict_object_control.json'
with open(output_filename, 'w') as f:
    json.dump(json_compatible_dict, f, indent=2)

In [11]:
play_indexed_dict.keys()

dict_keys(['20250618_113931_object_control_subj_1-2_and_6-1.1.videoTimeStamps', '20250618_113931_object_control_subj_1-2_and_6-1.2.videoTimeStamps', '20250618_123431_object_control_subj_1-3_and_6-3.1.videoTimeStamps', '20250618_123431_object_control_subj_1-3_and_6-3.2.videoTimeStamps', '20250618_133349_object_control_subj_4-1_and_4-2.1.videoTimeStamps', '20250618_133349_object_control_subj_4-1_and_4-2.2.videoTimeStamps', '20250618_143034_object-control-subj_4-3_and_4-4.1.videoTimeStamps', '20250618_143034_object-control-subj_4-3_and_4-4.2.videoTimeStamps', '22_44_object.1.videoTimeStamps', '22_44_object.2.videoTimeStamps', '22_object.videoTimeStamps', '23_41_object.1.videoTimeStamps', '23_41_object.2.videoTimeStamps', '23_object.videoTimeStamps', '31_32_object.1.videoTimeStamps', '31_32_object.2.videoTimeStamps', '31_object.videoTimeStamps', '32_object.videoTimeStamps', '41_object.videoTimeStamps', '44_object.videoTimeStamps'])